<a href="https://colab.research.google.com/github/justorfc/Estadistica_Aplicada_con_Python_y_R_2026_2/blob/main/14_Semana_14_Autocorrelaci%C3%B3n%2C_Memoria_del_Sistema_y_Pron%C3%B3stico_B%C3%A1sico.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Este es Notebook de la propuesta estructurada para la **Semana 14**, continuando dentro del **Eje V**. En esta semana, pasamos de simplemente describir una serie de tiempo (descomposición) a entender su "memoria" matemática y sentar las bases metodológicas para predecir el futuro (pronóstico o *forecasting*).

# Semana 14: Autocorrelación, Memoria del Sistema y Pronóstico Básico

**Resultado de aprendizaje:** Analiza la autocorrelación (ACF/PACF) de variables agroclimáticas, comprende la partición secuencial de datos temporales y evalúa modelos de pronóstico utilizando métricas específicas (MAE, RMSE, MAPE).

---

#### Sesión 1: La Memoria del Agua y Gráficos de Autocorrelación (80 - 90 minutos)

**Objetivo:** Comprender que en series de tiempo, el valor de hoy depende fuertemente del valor de ayer (autocorrelación), y visualizar esta "memoria" matemática usando Python.

* **20 min - Diálogo socrático y conceptualización (Lápiz y papel):**
* *Situación:* Si el caudal de un río hoy está extremadamente alto por las lluvias, ¿es probable que mañana esté completamente seco? No, el río tiene "memoria" (inercia hidrológica).
* *Actividad:* Se explica el concepto de rezago (*lag*). Los estudiantes dibujan qué significa relacionar el Caudal en el tiempo $t$ con el Caudal en el tiempo $t-1$. Introducción intuitiva a la Función de Autocorrelación (ACF) y Autocorrelación Parcial (PACF).


* **45 min - Exploración en Google Colab (Python):**
* Carga del cuaderno de la semana 14.
* Análisis exploratorio visual de la inercia de una serie de caudales.
* Uso de `statsmodels.graphics.tsaplots` para graficar el ACF y PACF.
* Interpretación de las bandas de confianza: ¿Hasta cuántos meses "recuerda" el río su estado anterior?


* **15 min - Reflexión manuscrita:**
* Interpretación técnica: ¿Por qué en una serie estacional (ej. lluvias) el gráfico ACF vuelve a subir cada 12 rezagos (meses)?



---

#### Sesión 2: Partición Temporal y Métricas de Pronóstico en R (80 - 90 minutos)

**Objetivo:** Comprender que en series de tiempo no podemos mezclar los datos aleatoriamente para validar (como hicimos en regresión múltiple), y trasladar el análisis de autocorrelación a R.

* **25 min - El riesgo de viajar en el tiempo (Data Leakage):**
* Explicación en pizarra de la validación temporal: El conjunto de Entrenamiento (Train) debe ser estrictamente el pasado, y el de Prueba (Test) estrictamente el futuro. Prohibido usar `train_test_split` aleatorio.
* Introducción a la métrica MAPE (Error Porcentual Absoluto Medio), vital para explicarle errores a gerentes o agricultores de forma intuitiva.


* **20 min - Prompts para autocorrelación en R:**
* Demostración de cómo instruir a la IA para calcular autocorrelación en R usando la librería `forecast` o `feasts`. El comando `ggAcf()` genera gráficos muy superiores estéticamente.


* **40 min - Reto en Posit Cloud:**
* Los estudiantes ejecutan el flujo en RMarkdown. Particionan una serie usando la función `window()`, grafican la autocorrelación de la serie de entrenamiento y documentan el proceso en su bitácora de IA.



---

A continuación, el contenido listo para integrarse en las celdas de tu cuaderno de Google Colab.

---

### Celda de Texto 1

# Semana 14: Autocorrelación y Partición Temporal
**Asignatura:** Estadística Aplicada con Python y R  
**Programa:** Ingeniería Agrícola - Universidad de Sucre  
**Profesor:** Justo Rafael Fuentes Cuello  

---

### Situación de Interés: La Memoria del Sistema Hidrológico
En la estadística tradicional (como en la regresión múltiple de semanas anteriores), asumíamos que cada dato era independiente del otro. En hidrología y agricultura, eso es falso.

Si el suelo está saturado hoy, es muy probable que siga húmedo mañana. A esto se le llama **Inercia o Memoria**. Para pronosticar cuánto caudal tendrá un río el próximo mes, el mejor indicador suele ser el caudal que tiene este mes. Hoy mediremos matemáticamente esa memoria usando las **Funciones de Autocorrelación (ACF y PACF)**.

```

### Celda de Código 1

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

sns.set_theme(style="ticks")
np.random.seed(2026)

# Simulamos una serie de Caudales Mensuales (m3/s) con fuerte memoria (Autoregresiva)
fechas = pd.date_range(start='2010-01-01', periods=150, freq='ME')

# Generamos un proceso autoregresivo: el caudal de hoy depende del 70% del caudal de ayer
caudal = np.zeros(150)
caudal[0] = 50 # Caudal inicial

for t in range(1, 150):
    # El 70% del caudal anterior + un ciclo estacional + lluvia aleatoria
    caudal[t] = 0.7 * caudal[t-1] + 10 * np.sin(2 * np.pi * t / 12) + np.random.normal(15, 5)

df_rio = pd.DataFrame({'Fecha': fechas, 'Caudal_m3s': caudal})
df_rio.set_index('Fecha', inplace=True)

# Graficamos la serie
plt.figure(figsize=(10, 4))
plt.plot(df_rio.index, df_rio['Caudal_m3s'], color='teal', lw=2)
plt.title('Caudal Mensual Simulado (Demostración de Inercia)')
plt.xlabel('Año')
plt.ylabel('Caudal (m³/s)')
plt.show()

### 1. Cuantificando la Memoria: ACF y PACF
El gráfico ACF (Autocorrelation Function) mide la correlación de la serie consigo misma, desfasada en el tiempo (rezagos o *lags*).
*   **Lag 1:** Correlación entre el mes actual y el mes anterior.
*   **Lag 12:** Correlación entre el mes actual y el mismo mes del año pasado.

La franja sombreada azul representa el intervalo de confianza (usualmente 95%). Si una barra sobresale de esa franja, significa que la correlación es estadísticamente significativa (no es producto del azar).

```

### Celda de Código 2

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Gráfico ACF (Autocorrelación Total)
# Evaluamos hasta 36 meses atrás (3 años)
plot_acf(df_rio['Caudal_m3s'], lags=36, ax=axes[0], color='teal')
axes[0].set_title('Función de Autocorrelación (ACF)')
axes[0].set_xlabel('Rezagos (Meses)')
axes[0].set_ylabel('Correlación')

# Gráfico PACF (Autocorrelación Parcial)
# Mide el efecto directo aislando los meses intermedios
plot_pacf(df_rio['Caudal_m3s'], lags=36, ax=axes[1], color='darkorange')
axes[1].set_title('Autocorrelación Parcial (PACF)')
axes[1].set_xlabel('Rezagos (Meses)')

plt.tight_layout()
plt.show()

### 2. Partición Temporal: La regla de oro del Pronóstico
En la Semana 12 hicimos partición de datos (Train/Test) eligiendo filas al azar. **¡Si haces eso en una serie de tiempo, estás haciendo trampa matemática!** Le estarías dando al modelo datos del futuro para predecir el pasado.

En series de tiempo, la partición debe ser estrictamente secuencial: Entrenamos con el pasado (ej. primeros 120 meses) y Validamos con el futuro (ej. últimos 30 meses).

```

### Celda de Código 3

In [ ]:
# Definimos el punto de corte (80% de los datos)
punto_corte = int(len(df_rio) * 0.8)

# Partición estrictamente secuencial
train = df_rio.iloc[:punto_corte]
test = df_rio.iloc[punto_corte:]

print(f"Total datos históricos: {len(df_rio)}")
print(f"Meses para Entrenamiento (Pasado): {len(train)} (Hasta {train.index[-1].strftime('%Y-%m')})")
print(f"Meses para Validación (Futuro a predecir): {len(test)} (Desde {test.index[0].strftime('%Y-%m')})")

# Visualización de la partición
plt.figure(figsize=(10, 4))
plt.plot(train.index, train['Caudal_m3s'], label='Entrenamiento (Train)', color='teal')
plt.plot(test.index, test['Caudal_m3s'], label='Prueba (Test)', color='orange')
plt.axvline(train.index[-1], color='red', linestyle='--', label='Punto de Corte')
plt.title('Partición Secuencial de la Serie de Tiempo')
plt.ylabel('Caudal (m³/s)')
plt.legend()
plt.show()

### 3. Modelo Ingenuo de Pronóstico y Métricas (MAPE)
Antes de construir modelos complejos matemáticos (como ARIMA, que veremos en la Semana 15), siempre se construye un modelo de "línea base".
Un modelo común es el "Ingenuo Estacional": asume que el caudal de este mes será exactamente igual al caudal del mismo mes del año pasado.

Evaluaremos su desempeño con el **MAPE (Mean Absolute Percentage Error)**, que nos da el error en un porcentaje fácil de interpretar para cualquier gerente.

```

### Celda de Código 4

In [ ]:
# Modelo Ingenuo Estacional (Seasonal Naive)
# Tomamos el último año de entrenamiento y lo repetimos hacia el futuro
pronostico_naive = train['Caudal_m3s'].iloc[-12:].values
# Como el test set tiene 30 meses, repetimos este patrón
pronostico_naive = np.tile(pronostico_naive, 3)[:len(test)]

# Calculamos las métricas
def calcular_mape(y_real, y_pred):
    return np.mean(np.abs((y_real - y_pred) / y_real)) * 100

from sklearn.metrics import mean_absolute_error, mean_squared_error
mae = mean_absolute_error(test['Caudal_m3s'], pronostico_naive)
rmse = np.sqrt(mean_squared_error(test['Caudal_m3s'], pronostico_naive))
mape = calcular_mape(test['Caudal_m3s'], pronostico_naive)

print("--- Evaluación del Pronóstico de Línea Base ---")
print(f"MAE:  {mae:.2f} m³/s")
print(f"RMSE: {rmse:.2f} m³/s")
print(f"MAPE: {mape:.1f}% (El modelo se equivoca en promedio un {mape:.1f}%)")

# Gráfico del pronóstico vs realidad
plt.figure(figsize=(10, 4))
plt.plot(train.index[-24:], train['Caudal_m3s'].iloc[-24:], label='Pasado Reciente (Train)', color='teal')
plt.plot(test.index, test['Caudal_m3s'], label='Realidad Oculta (Test)', color='gray', alpha=0.5)
plt.plot(test.index, pronostico_naive, label='Pronóstico Ingenuo', color='red', linestyle='--')
plt.title('Pronóstico Base vs. Realidad Futura')
plt.legend()
plt.show()

### 🛑 Reflexión y Reserva Cognitiva (Síntesis manuscrita)
Toma tu lápiz y bitácora, y analiza la metodología empleada hoy:
1. Revisa el gráfico ACF de los caudales. ¿Hasta qué mes (rezago) observas que la barra azul sobrepasa la franja sombreada? ¿Qué te dice esto sobre cuánto tiempo el río "recuerda" un evento extremo de lluvia?
2. En el gráfico ACF, se notan picos que vuelven a subir en los rezagos 12, 24 y 36. ¿Por qué ocurre esto matemáticamente en variables como caudales o lluvias?
3. El error MAPE calculado fue de un porcentaje específico. Si estuvieras diseñando la política de entrega de agua de riego de un embalse, ¿considerarías que un error porcentual de esa magnitud en el pronóstico es seguro o construirías un modelo más avanzado?

---

### Instrucciones para el reto en R (Trabajo Autónomo y Sesión 2)

**Misión:** Explorar la "memoria" de una serie es visualmente muy superior en R gracias al paquete `forecast` y su integración con `ggplot2`. Tu reto es particionar la serie y generar los gráficos ACF en **Posit Cloud**.

**Pasos a seguir:**
1. Abre tu proyecto en Posit Cloud y crea un nuevo documento RMarkdown/Quarto.
2. Utiliza este *prompt* con tu asistente de IA (ChatGPT, Gemini, Claude):
   > *"Actúa como un profesor de hidrología estadística usando R. En mi clase con Python, simulé una serie de caudales mensuales y utilicé statsmodels para generar gráficos ACF y PACF. Luego, particioné la serie secuencialmente (80% Train, 20% Test) para validación temporal. Necesito replicar esto en R. Escribe código para generar la serie simulada y convertirla en un objeto `ts`. Muéstrame cómo usar el comando `window()` para hacer la partición temporal sin usar muestreo aleatorio. Luego, usa la librería `forecast` (específicamente `ggAcf` y `ggPacf`) para hacer los gráficos de autocorrelación sobre el conjunto de entrenamiento. Explica los pasos para documentarlos en mi RMarkdown."*
3. Observa cómo la función `window()` de R está diseñada nativamente para extraer rangos de tiempo (ej. de 2010 a 2021) respetando la estructura de la serie, a diferencia de Python donde cortamos usando índices numéricos.
4. **Entrega:** Renderiza tu documento. En tu "Bitácora de IA", incluye una evaluación de los gráficos generados por `ggAcf`. ¿Te parecieron más fáciles de interpretar visualmente que los de Python (`statsmodels`)? Justifica tu respuesta.

```